In [1]:
from dotenv import load_dotenv
import os

from agents import Agent, Runner

In [6]:
load_dotenv(override=True)

True

In [7]:
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

In [8]:
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

In [10]:
first_agent = Agent(name="first agent", instructions="You are to all general questions", model="gpt-4o-mini")

In [13]:
first_runner = await Runner.run(first_agent, "Who is the current US President")

[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: sk-or-v1*************************************************************ac47. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "code": "invalid_api_key",
    "param": null
  },
  "status": 401
}


[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: sk-or-v1*************************************************************ac47. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "code": "invalid_api_key",
    "param": null
  },
  "status": 401
}


In [14]:
print(first_runner.final_output)

As of my last update in October 2023, the current President of the United States is Joe Biden. He was inaugurated on January 20, 2021. Please verify with up-to-date sources, as my information may not reflect the most current events.


In [26]:
from IPython.display import Markdown, display
from pypdf import PdfReader
import gradio as gr
from dotenv import load_dotenv
import os

In [27]:
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

from agents import Agent, Runner

os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

In [28]:
pdfReader = PdfReader("Resources/Profile.pdf")
prof_summary = " "
for page in pdfReader.pages:
    text = page.extract_text()
    if text:
        prof_summary += text + "\n"

In [29]:
prof_summary

" Parag Agrawal - Professional Profile\nSummary\nParag Agrawal is an Indian-American software engineer and entrepreneur, best known for his tenure\nas the CEO of Twitter from November 2021 to October 2022. He was appointed CEO following Jack\nDorsey's resignation but was dismissed after Elon Musk's acquisition of Twitter.\nEducation and Early Career\nAgrawal holds a B.Tech in Computer Science and Engineering from the Indian Institute of\nTechnology (IIT) Bombay. He then pursued a Ph.D. in Computer Science at Stanford University,\nwhere his research focused on uncertainty in data management and integration. Prior to his role at\nTwitter, he held research internships at Microsoft Research and Yahoo! Research.\nCareer at Twitter\nAgrawal joined Twitter in 2011 as a software engineer and quickly rose through the ranks. In 2017,\nhe was appointed Chief Technology Officer (CTO), overseeing Twitter's technical strategy and\nleading initiatives like Project Bluesky, aimed at developing a decen

In [30]:
name = "Parag Agrawal"

system_prompt = (
    f"You are acting as {name}, representing {name} on their website. "
    f"Your role is to answer questions specifically about {name}'s career, background, skills, and experience. "
    f"You must faithfully and accurately portray {name} in all interactions. "
    f"You have access to a detailed summary of {name}'s background and their LinkedIn profile, which you should use to inform your answers. "
    f"Maintain a professional, engaging, and approachable tone, as if you are speaking to a potential client or future employer visiting the site. "
    f"If you are unsure of an answer, it is better to honestly acknowledge that than to guess."
    f" LinkedIn Profile:\n{prof_summary}\n\n"
    f"Using this context, please converse naturally and consistently, always staying in character as {name}."
)

In [31]:
chatbot_agent = Agent(
    name="Portfolio chatbot",
    instructions=system_prompt,
    model="gpt-4o-mini"
)

In [32]:
async def liveChat(message, history):

    messages = [{"role": msg["role"], "content": msg["content"]} for msg in history]
    messages.append({"role": "user", "content": message})
    result = await Runner.run(chatbot_agent, messages)
    return result.final_output

In [ ]:
gr.ChatInterface(
    liveChat,
    title="Chat with Parag",
    description="Ask me about anything, from career to background and skills"
).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: sk-or-v1*************************************************************ac47. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "code": "invalid_api_key",
    "param": null
  },
  "status": 401
}
[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: sk-or-v1*************************************************************ac47. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "code": "invalid_api_key",
    "param": null
  },
  "status": 401
}


In [1]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# Disable tracing BEFORE importing agents
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

# Configure for OpenRouter
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

from agents import Agent, Runner, function_tool

# Define a tool using the @tool decorator
@function_tool
def get_weather(city: str) -> str:
    """
    Get the current weather for a city.
    
    Args:
        city: The name of the city to get weather for
    """
    weather_data = {
        "london": "Cloudy, 15°C",
        "tokyo": "Sunny, 22°C",
        "new york": "Rainy, 18°C"
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")

# Create agent with the tool
weather_agent = Agent(
    name="WeatherBot",
    instructions="You help users check the weather. Use the get_weather tool when asked about weather.",
    model="gpt-4o-mini",
    tools=[get_weather]
)

# Run it
runner = await Runner.run(weather_agent, "What's the weather in Tokyo?")
print(runner.final_output)

The weather in Tokyo is sunny with a temperature of 22°C.


In [7]:
@function_tool
def get_user_id(email: str) -> str:
    """Look up user ID from email."""
    return "user_123"

@function_tool
def get_user_orders(user_id: str) -> str:
    """Get orders for a user ID."""
    return "Order #1: Laptop, Order #2: Mouse"

@function_tool
def get_order_status(order_id: str) -> str:
    """Get status of an order."""
    return "Shipped - arriving tomorrow"

agent = Agent(
    name="OrderAssistant",
    instructions="Help users check their order status. First find their user ID, then their orders, then status.",
    model="gpt-4o-mini",
    tools=[get_user_id, get_user_orders, get_order_status]
)

runner = await Runner.run(agent, "What's the status of my orders? My email is john@example.com")
print(runner.final_output)

The status of your orders is as follows:

- **Laptop**: Shipped - arriving tomorrow
- **Mouse**: Shipped - arriving tomorrow


In [6]:
@function_tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"{city}: Sunny, 25°C"

@function_tool
def get_news(topic: str) -> str:
    """Get latest news on a topic."""
    return f"Latest {topic} news: ..."

@function_tool
def get_stock(symbol: str) -> str:
    """Get stock price."""
    return f"{symbol}: $150.00"

agent = Agent(
    name="MorningBriefing",
    instructions="Provide a morning briefing with weather, news, and stock info.",
    model="gpt-4o-mini",
    tools=[get_weather, get_news, get_stock]
)

runner = await Runner.run(agent, "Give me my morning briefing for NYC, tech news, and AAPL")
print(runner.final_output)

# User: "Give me my morning briefing for NYC, tech news, and AAPL"
# Agent calls all 3 tools in parallel → Compiles response

### Morning Briefing

**Weather in New York City:**
- **Condition:** Sunny
- **Temperature:** 25°C

**Latest Tech News:**
- [Details on the specific tech news articles will be included here.]

**Apple Inc. (AAPL) Stock Information:**
- **Current Price:** $150.00

Have a great day! Let me know if you need more information.


In [5]:
@function_tool
def search_web(query: str) -> str:
    """Search the web for current information."""
    return f"Web results for '{query}': ..."

@function_tool
def search_database(query: str) -> str:
    """Search internal company database."""
    return f"Database results for '{query}': ..."

@function_tool
def search_documents(query: str) -> str:
    """Search uploaded documents."""
    return f"Document results for '{query}': ..."

agent = Agent(
    name="SmartSearch",
    instructions="""You are a search assistant.
    
    Choose the right search based on the query:
    - For current events/general info → use search_web
    - For company/employee info → use search_database
    - For policy/procedure questions → use search_documents
    """,
    model="gpt-4o-mini",
    tools=[search_web, search_database, search_documents]
)

runner = await Runner.run(agent, "Find the latest company policy on remote work.")
print(runner.final_output)

It seems that I wasn't able to retrieve any specific documents regarding the latest company policy on remote work. Please check your internal documents or company intranet for the most up-to-date information on remote work policies. If you need help with anything else, let me know!


In [10]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

from agents import Agent, Runner, function_tool
import requests

@function_tool
def get_weather(city: str) -> str:
    """
    Get the current weather for a city.
    
    Args:
        city: The name of the city to get weather for
    """
    try:
        url = f"https://wttr.in/{city}?format=%C+%t+%h+%w"
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            return f"Weather in {city}: {response.text}"
        return f"Could not fetch weather for {city}"
    except Exception as e:
        return f"Error: {str(e)}"

@function_tool
def search_wikipedia(query: str) -> str:
    """
    Search Wikipedia for information.
    
    Args:
        query: The topic to search for
    """
    try:
        url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{query}"
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            return data.get("extract", "No information found")
        return f"No Wikipedia article found for {query}"
    except Exception as e:
        return f"Error: {str(e)}"

@function_tool
def get_random_joke() -> str:
    """Get a random joke."""
    try:
        response = requests.get(
            "https://official-joke-api.appspot.com/random_joke",
            timeout=10
        )
        if response.status_code == 200:
            data = response.json()
            return f"{data['setup']} - {data['punchline']}"
        return "Could not fetch joke"
    except Exception as e:
        return f"Error: {str(e)}"


assistant = Agent(
    name="RealWorldAssistant",
    instructions="""You are a helpful assistant with real-world capabilities.

You can:
- Get LIVE weather for any city using get_weather
- Search Wikipedia for information using search_wikipedia  
- Tell jokes using get_random_joke

Always use your tools when asked about weather, facts, or jokes.
Be friendly and informative.""",
    model="gpt-4o-mini",
    tools=[get_weather, search_wikipedia, get_random_joke]
)


import asyncio

async def main():
    print("=== Weather Test ===")
    runner = await Runner.run(assistant, "What's the weather in Lagos, Nigeria?")
    print(runner.final_output)

    print("\n=== Wikipedia Test ===")
    runner = await Runner.run(assistant, "Tell me about Python programming language")
    print(runner.final_output)

    print("\n=== Joke Test ===")
    runner = await Runner.run(assistant, "Tell me a joke")
    print(runner.final_output)

if __name__ == "__main__":
    await main()

=== Weather Test ===
It seems that I am currently unable to access the weather information for Lagos, Nigeria, due to a connection issue. You might want to check a weather website or app for the latest updates. If there's anything else I can help you with, feel free to ask!

=== Wikipedia Test ===
It seems I'm having trouble finding information from Wikipedia about the Python programming language. However, I can provide you with a brief overview:

Python is a high-level, interpreted programming language known for its easy readability and simplicity. It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming.

Key features of Python include:
- **Easy to Learn:** Python's syntax is designed to be intuitive and accessible for beginners.
- **Versatile:** It is used in various fields such as web development, data analysis, artificial intelligence, scientific computing, and m

In [23]:
from agents import Agent, Runner, handoff

# Specialist agents
billing_agent = Agent(
    name="BillingSpecialist",
    instructions="""You handle billing questions:
    - Payment issues
    - Refunds
    - Invoice requests
    Be helpful and resolve issues quickly."""
)

technical_agent = Agent(
    name="TechnicalSupport",
    instructions="""You handle technical issues:
    - Bug reports
    - How-to questions
    - Feature requests
    Ask clarifying questions to understand the issue."""
)

# Triage agent that routes to specialists
triage_agent = Agent(
    name="Triage",
    instructions="""You are the first point of contact. Route customers immediately:
- Billing/payment/charge issues → transfer to BillingSpecialist RIGHT AWAY
- Technical problems → transfer to TechnicalSupport RIGHT AWAY

Do NOT ask clarifying questions if the intent is already clear. Transfer immediately.""",
    model="gpt-4o-mini",
    handoffs=[
        handoff(billing_agent),
        handoff(technical_agent)
    ]
)

# Run the triage agent
result = await Runner.run(triage_agent, "I was charged twice for my subscription!")
print(result.final_output)
# → Will handoff to billing_agent

Hi! I'm here to help with your billing questions. Could you please tell me more about your issue—are you experiencing a payment problem, need a refund, or want to request an invoice? Let me know the details, and I'll assist you right away!


In [22]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

from agents import Agent, Runner, function_tool

# Create specialist agents
researcher = Agent(
    name="Researcher",
    instructions="You research topics thoroughly and return detailed findings.",
    model="gpt-4o"
)

writer = Agent(
    name="Writer", 
    instructions="You write clear, engaging content based on provided information.",
    model="gpt-4o"
)

# tool functions that use agents
@function_tool
async def research_topic(topic: str) -> str:
    """
    Research a topic thoroughly.
    
    Args:
        topic: The topic to research
    """
    result = await Runner.run(researcher, f"Research this topic: {topic}")
    return result.final_output

@function_tool
async def write_content(brief: str) -> str:
    """
    Write content based on a brief.
    
    Args:
        brief: The writing brief with topic and key points
    """
    result = await Runner.run(writer, brief)
    return result.final_output

# Orchestrator that uses other agents as tools
orchestrator = Agent(
    name="ContentManager",
    instructions="""You manage content creation.
    
    When asked to create content:
    1. Use research_topic to gather information
    2. Use write_content to create the final piece
    3. Review and present the result
    """,
    model="gpt-4o-mini",
    tools=[research_topic, write_content]
)

# Usage
runner = await Runner.run(
    orchestrator,
    "Create a blog post about the benefits of meditation"
)
print(runner.final_output)

### Unlocking the Power of Meditation: Discover the Profound Benefits

In today's fast-paced world, stress seems to be an unwelcome companion for many. Whether it's due to work, personal life, or the constant bombardment of information, finding a moment of peace can be elusive. Enter meditation — an ancient practice that has stood the test of time, promising profound benefits to those who embrace it regularly. From stress reduction to emotional well-being, let's explore the myriad ways meditation can enhance your life.

#### 1. Stress Reduction

Perhaps the most well-known benefit, meditation acts as a powerful antidote to stress. By focusing your mind and letting go of the chaos, meditation can significantly lower cortisol levels, the hormone associated with stress. A calm mind leads to a calm body, allowing you to navigate life's challenges with grace and ease.

#### 2. Anxiety Control

Closely linked to stress reduction, meditation provides a space to observe your thoughts without j

In [24]:
from dotenv import load_dotenv
import os
from winotify import Notification, audio
import threading
from datetime import datetime, timedelta
import re

# toaster = ToastNotifier()

load_dotenv(override=True)

os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

In [25]:
from agents import Agent, Runner, function_tool
from datetime import datetime

# In-memory storage (use a database in production)
todos = []
reminders = []

In [26]:
@function_tool
def add_todo(task: str, priority: str = "medium") -> str:
    """
    Add a task to the todo list.
    
    Args:
        task: The task description
        priority: Priority level (low, medium, high)
    """
    todo = {"task": task, "priority": priority, "done": False, "id": len(todos) + 1}
    todos.append(todo)
    return f"Added: '{task}' with {priority} priority (ID: {todo['id']})"

@function_tool
def list_todos() -> str:
    """List all todos."""
    if not todos:
        return "No todos yet!"
    return "\n".join([
        f"[{'✓' if t['done'] else ' '}] {t['id']}. {t['task']} ({t['priority']})"
        for t in todos
    ])

@function_tool
def complete_todo(todo_id: int) -> str:
    """
    Mark a todo as complete.
    
    Args:
        todo_id: The ID of the todo to complete
    """
    for todo in todos:
        if todo['id'] == todo_id:
            todo['done'] = True
            return f"Completed: '{todo['task']}'"
    return f"Todo {todo_id} not found"


def parse_time(time_str: str) -> datetime:
    """Parse time string like 'in 5 minutes', 'in 1 hour', 'at 3pm'"""
    now = datetime.now()
    time_str = time_str.lower().strip()
    
    # Handle "in X minutes/hours"
    if "in" in time_str:
        match = re.search(r'in\s+(\d+)\s*(minute|min|hour|hr|second|sec)', time_str)
        if match:
            amount = int(match.group(1))
            unit = match.group(2)
            if 'min' in unit:
                return now + timedelta(minutes=amount)
            elif 'hour' in unit or 'hr' in unit:
                return now + timedelta(hours=amount)
            elif 'sec' in unit:
                return now + timedelta(seconds=amount)
    
    # Handle "at 3pm", "at 15:00"
    if "at" in time_str:
        match = re.search(r'at\s+(\d{1,2})(?::(\d{2}))?\s*(am|pm)?', time_str)
        if match:
            hour = int(match.group(1))
            minute = int(match.group(2)) if match.group(2) else 0
            period = match.group(3)
            
            if period == 'pm' and hour != 12:
                hour += 12
            elif period == 'am' and hour == 12:
                hour = 0
            
            target = now.replace(hour=hour, minute=minute, second=0)
            if target <= now:
                target += timedelta(days=1)
            return target
    
    # Default: 1 minute from now
    return now + timedelta(minutes=1)

# Replace the send_notification function
def send_notification(message: str, delay_seconds: float):
    """Send notification after delay"""
    def notify():
        threading.Event().wait(delay_seconds)
        toast = Notification(
            app_id="Personal Assistant",
            title="⏰ Reminder",
            msg=message,
            duration="short"
        )
        toast.set_audio(audio.Default, loop=False)
        toast.show()
    
    thread = threading.Thread(target=notify)
    thread.daemon = True
    thread.start()

@function_tool
def set_reminder(message: str, time: str) -> str:
    """
    Set a reminder that sends a Windows notification.
    
    Args:
        message: What to be reminded about
        time: When to be reminded (e.g., "in 5 minutes", "in 1 hour", "at 3pm")
    """
    target_time = parse_time(time)
    delay = (target_time - datetime.now()).total_seconds()
    
    if delay < 0:
        delay = 0
    
    reminder = {
        "message": message, 
        "time": time, 
        "target": target_time.isoformat(),
        "created": datetime.now().isoformat()
    }
    reminders.append(reminder)
    
    # Schedule the notification
    send_notification(message, delay)
    
    return f"✅ Reminder set: '{message}' for {time} (will notify at {target_time.strftime('%H:%M:%S')})"


@function_tool
def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [27]:
# Create the assistant
assistant = Agent(
    name="PersonalAssistant",
    instructions="""You are a helpful personal assistant.
    
    You can help with:
    - Managing todos (add, list, complete)
    - Setting reminders
    - Telling the time
    
    Be friendly and proactive. If the user adds a todo, 
    ask if they want to set a reminder for it.
    """,
    model="gpt-4o-mini",
    tools=[add_todo, list_todos, complete_todo, set_reminder, get_current_time]
)

In [28]:
async def chat():
    print("Personal Assistant ready! Type 'quit' to exit.\n")
    
    while True:
        user_input = input("You: ")
        if user_input.lower() == 'quit':
            break
        
        runner = await Runner.run(assistant, user_input)
        print(f"Assistant: {runner.final_output}\n")

await chat()

Personal Assistant ready! Type 'quit' to exit.

Assistant: The current time is 15:20 (3:20 PM). How can I assist you further?

